In [1]:
import pandas as pd 
import numpy as np 
import ast 
import nltk
import csv
import os
import re

In [2]:


def fix_and_load_csv(file_name):
    fixed_rows = []
    with open(file_name, 'r', encoding='utf-8') as f:
        reader = csv.reader(f)
        header = next(reader)
        
        for row in reader:
            if len(row) < 9: continue
            
            # 1. Base columns (place to avg_cost)
            res = row[:8]
            
            # 2. Extract Map Link (Column 8)
            raw_8 = row[8].strip()
            url_match = re.search(r'(https?://[^\s,]+)', raw_8)
            url = url_match.group(1) if url_match else raw_8
            
            # 3. Detect Route and Security (The tricky part)
            route_in_8 = raw_8.replace(url, "").strip().strip(',')
            
            if route_in_8:
                # Case A: Link and Route were merged into Column 8
                route = route_in_8
                security = row[9] if len(row) > 9 else "Standard safety"
                restaurants = ", ".join(row[10:]) if len(row) > 10 else "Local food"
            else:
                # Case B: Link is in Column 8, Route is in Column 9
                route = row[9] if len(row) > 9 else "Check description"
                security = row[10] if len(row) > 10 else "Standard safety"
                restaurants = ", ".join(row[11:]) if len(row) > 11 else "Local food"
            
            # Add them to the result
            res.extend([url, route, security, restaurants])
            fixed_rows.append(res)
            
    return pd.DataFrame(fixed_rows, columns=['place','type','district','budget','description','best_time','weather','avg_cost','map_link','route','security','nearby restaurants'])

In [3]:
df_ctg = fix_and_load_csv('chattagram.csv')
df_dhaka = fix_and_load_csv('dhaka.csv')
df_barisal = fix_and_load_csv('barisal.csv')
df_cumilla = fix_and_load_csv('cumilla.csv')
df_khulna = fix_and_load_csv('khulna.csv')
df_rajshahi = fix_and_load_csv('rajshahi.csv')
df_rangpur = fix_and_load_csv('rangpur.csv')
df_sylhet = fix_and_load_csv('sylhet.csv')


##example-------------------------
df_sylhet.head(1)
df_rajshahi.head(1)


,place,type,district,budget,description,best_time,weather,avg_cost,map_link,route,security,nearby restaurants
0,Puthia Temple Complex,Religious & Historical,Rajshahi,Low,Ancient Hindu temple complex with terracotta p...,Nov-Feb,Cool and dry,150-400,https://www.google.com/maps/search/Puthia+Temp...,Bus from Rajshahi city (30-45 min),Generally safe temple complex with visitors.,"Puthia local restaurants, Temple area food"


In [4]:
df_ctg.isnull().sum()
df_sylhet.isnull().sum()
df_dhaka.isnull().sum()
df_cumilla.isnull().sum()

place                 0
type                  0
district              0
budget                0
description           0
best_time             0
weather               0
avg_cost              0
map_link              0
route                 0
security              0
nearby restaurants    0
dtype: int64

In [5]:
df_ctg.duplicated().sum()
df_dhaka.duplicated().sum()
df_sylhet.duplicated().sum()
df_cumilla.duplicated().sum()
df_rangpur.duplicated().sum()
df_khulna.duplicated().sum()
df_rajshahi.duplicated().sum()
df_barisal.duplicated().sum()

np.int64(0)

In [6]:
def Clean_data(df):
     ##for safety to handle different CSV header styles
     df.columns = df.columns.str.lower().str.strip()
     df= df[['place','type','district','budget','description','best_time','weather','avg_cost','map_link','route','security','nearby restaurants']]

    ##remove spaces  + convert lowercase

     df['description'] = df['description'].astype(str).str.lower().str.strip()
     df['type'] = df['type'].astype(str).str.lower().str.strip()
     df['district'] = df['district'].astype(str).str.lower().str.strip() # Clean the district too!
     df['budget'] = df['budget'].astype(str).str.lower().str.strip()
     df['weather'] = df['weather'].astype(str).str.lower().str.strip()
     # This splits the link at the first comma and takes only the URL part
     df['map_link'] = df['map_link'].apply(lambda x: x.split(',')[0].strip())

     df = df.replace('nan', '')


     ##create a new column name : tags => description+type+budget+weather

     df['tags']= df['description']+" "+ df['type']+" "+ df['budget']+ " "+df['weather']
     df = df[['place','type','district','budget','best_time','avg_cost','map_link','route','description','weather','nearby restaurants','security','tags']]

     return df



##now apply the function
df_ctg= Clean_data(df_ctg)
df_dhaka= Clean_data(df_dhaka)
df_sylhet= Clean_data(df_sylhet)
df_cumilla= Clean_data(df_cumilla)
df_barisal= Clean_data(df_barisal)
df_rangpur= Clean_data(df_rangpur)
df_rajshahi= Clean_data(df_rajshahi)
df_khulna= Clean_data(df_khulna)
   

     
       
        

In [7]:
##now merge all of them

new_df= pd.concat([df_ctg,df_dhaka,df_sylhet,df_barisal,df_cumilla,df_rangpur,df_rajshahi,df_khulna], ignore_index=True)
new_df.head()

,place,type,district,budget,best_time,avg_cost,map_link,route,description,weather,nearby restaurants,security,tags
0,Patenga Sea Beach,beach,chattogram,low,Nov-Feb,200-400,https://www.google.com/maps/search/Patenga+Sea...,Bus or CNG from New Market (20-30 min),popular beach near airport with sunset views a...,pleasant cool winter,"Koyla Restaurant, The River View, Ambrosia, ...",Generally safe during day with crowds and poli...,popular beach near airport with sunset views a...
1,Foy's Lake,natural lake & amusement,chattogram,medium,Nov-Feb,400-800,https://www.google.com/maps/search/Foy's+Lake+...,Bus from GEC Circle or CNG (15 min),artificial lake with amusement park boating zo...,cool and dry,"Foy's Lake Resort Restaurant, Secret Recipe G...",Generally safe family spot with good security ...,artificial lake with amusement park boating zo...
2,Ethnological Museum,museum,chattogram,low,Year-round,100-200,https://www.google.com/maps/search/Ethnologica...,Rickshaw or bus from city center,cultural museum showcasing artifacts and lifes...,indoor comfortable,"Handi Restaurant, Mezban Haile Aiun, Koyla R...",Very safe indoor city venue.,cultural museum showcasing artifacts and lifes...
3,Shrine of Bayazid Bostami,religious site,chattogram,low,Year-round,50-150,https://www.google.com/maps/search/Bayazid+Bos...,CNG from Chawkbazar (10 min),historic sufi shrine with sacred turtles and p...,pleasant mornings,"Local shrine eateries, Chawkbazar mezban, Si...",Generally safe and peaceful religious site.,historic sufi shrine with sacred turtles and p...
4,Chittagong War Cemetery,historical,chattogram,low,Year-round,Free-100,https://www.google.com/maps/search/Chittagong+...,Short rickshaw from city center,well-maintained wwii cemetery with memorials a...,cool mornings,"Handi Restaurant, Local cafes",Very safe and serene low-crime area.,well-maintained wwii cemetery with memorials a...


In [8]:
# Run this in Jupyter to turn "200-400" into a single number (300) for calculation
def get_avg(cost_str):
    try:
        if '-' in str(cost_str):
            nums = [int(s) for s in cost_str.split('-') if s.isdigit()]
            return sum(nums) / len(nums)
        return int(cost_str)
    except:
        return 0

new_df['numeric_cost'] = new_df['avg_cost'].apply(get_avg)

In [9]:
from nltk.stem.porter import PorterStemmer
ps= PorterStemmer()


def stem(text):
    y=[]

    for i in text.split():
        y.append(ps.stem(i))

    return " ".join(y)


new_df['tags']= new_df['tags'].apply(stem)
new_df.tail()
    

,place,type,district,budget,best_time,avg_cost,map_link,route,description,weather,nearby restaurants,security,tags,numeric_cost
363,Satkhira Sundarbans Entry,wildlife,khulna,medium,Nov-Feb,500-1100,https://www.google.com/maps/search/Satkhira+Su...,Bus + boat,alternative sundarbans access point,dry,Satkhira local restaurants,Moderate to high caution; Sundarbans area—orga...,altern sundarban access point wildlif medium dri,800.0
364,Chuadanga Local Park,park,khulna,low,Year-round,Free-100,https://www.google.com/maps/search/Chuadanga+P...,Bus,small local green space,any,Chuadanga local food,Very safe small park.,small local green space park low ani,100.0
365,Kushtia Cultural Spots,cultural,khulna,low,Year-round,100-300,https://www.google.com/maps/search/Kushtia+nea...,Bus,local cultural and folk sites,any,Kushtia Lalon area eateries,Generally safe cultural spots.,local cultur and folk site cultur low ani,200.0
366,Bagerhat Mosque City Walk,historical,khulna,low,Year-round,Free-200,https://www.google.com/maps/search/Bagerhat+Mo...,Local,walking tour of multiple historic mosques,any,Bagerhat mosque city restaurants,Generally safe heritage walking area.,walk tour of multipl histor mosqu histor low ani,200.0
367,Mongla Port View,harbor view,khulna,low,Year-round,Free,https://www.google.com/maps/search/Mongla+Port,Bus from Khulna,bustling port area near sundarbans,any,Mongla port restaurants,Busy port area—standard precautions; avoid res...,bustl port area near sundarban harbor view low...,0.0


In [10]:
## now Tfid Vectorization + cosine similarity

from sklearn.feature_extraction.text import TfidfVectorizer
tfidf= TfidfVectorizer(max_features=5000,stop_words='english')

vectors=tfidf.fit_transform(new_df['tags']).toarray()

vectors.shape

(368, 482)

In [11]:
from sklearn.metrics.pairwise import cosine_similarity

similarity= cosine_similarity(vectors)
similarity.shape

(368, 368)

In [12]:

import joblib

# The 'compress' parameter accepts an integer from 0 to 9 (9 being the highest compression)
joblib.dump(new_df, 'main_df.joblib', compress=3)
joblib.dump(similarity, 'similarity.joblib', compress=3)
joblib.dump(tfidf, 'tfidf.joblib',compress=3)


['tfidf.joblib']